<a href="https://colab.research.google.com/github/Manoj132005/AU_Workshop_hackathon/blob/main/App_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ─────────────────────────────────────────────────────────────
# Page Config
# ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title = "🌫️ AQI Predictor",
    page_icon  = "🌿",
    layout     = "wide",
    initial_sidebar_state = "expanded"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 54.7 MB/s eta 0:00:00


2026-03-11 10:46:00.145 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [3]:
#CSS

In [4]:
st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Rajdhani:wght@500;600;700&family=Inter:wght@400;500;600&display=swap');

    html, body, [class*="css"] {
        font-family: 'Inter', sans-serif;
    }
    .main-title {
        font-family: 'Rajdhani', sans-serif;
        font-size: 3rem;
        font-weight: 700;
        background: linear-gradient(135deg, #11998e, #38ef7d);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 0;
    }
    .sub-title {
        color: #888;
        font-size: 1rem;
        margin-top: 0;
        margin-bottom: 2rem;
    }
    .aqi-card {
        padding: 28px 24px;
        border-radius: 16px;
        text-align: center;
        color: white;
        margin: 10px 0;
    }
    .aqi-value {
        font-family: 'Rajdhani', sans-serif;
        font-size: 5rem;
        font-weight: 700;
        line-height: 1;
    }
    .aqi-label {
        font-size: 1.4rem;
        font-weight: 600;
        margin-top: 4px;
    }
    .metric-box {
        background: #1a1a2e;
        border: 1px solid #2a2a4a;
        border-radius: 12px;
        padding: 18px;
        text-align: center;
        color: white;
    }
    .metric-val {
        font-family: 'Rajdhani', sans-serif;
        font-size: 2rem;
        font-weight: 700;
        color: #38ef7d;
    }
    .metric-lbl {
        font-size: 0.85rem;
        color: #aaa;
        margin-top: 4px;
    }
    .section-header {
        font-family: 'Rajdhani', sans-serif;
        font-size: 1.4rem;
        font-weight: 600;
        color: #38ef7d;
        border-bottom: 1px solid #2a2a4a;
        padding-bottom: 6px;
        margin: 20px 0 12px 0;
    }
    .stSlider > div > div { color: #38ef7d; }
    .sidebar .sidebar-content { background: #0d0d1a; }
    div[data-testid="stSidebar"] {
        background: linear-gradient(180deg, #0d0d1a 0%, #0a0a16 100%);
    }
</style>
""", unsafe_allow_html=True)

2026-03-11 10:46:36.619 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:46:36.994 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-11 10:46:36.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:46:36.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [5]:
#Load Model

In [6]:
@st.cache_resource
def load_model():
    with open("best_rf_aqi_model.pkl", "rb") as f:
        return pickle.load(f)

try:
    model = load_model()
    model_loaded = True
except:
    model_loaded = False

2026-03-11 10:47:06.131 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [7]:
def get_aqi_info(aqi):
    if   aqi <= 50:  return "Good",         "#2ecc71", "🟢", "Minimal impact on health."
    elif aqi <= 100: return "Satisfactory",  "#f1c40f", "🟡", "Minor discomfort to sensitive people."
    elif aqi <= 200: return "Moderate",      "#e67e22", "🟠", "Discomfort to asthma patients."
    elif aqi <= 300: return "Poor",          "#e74c3c", "🔴", "Breathing discomfort to most people."
    elif aqi <= 400: return "Very Poor",     "#8e44ad", "🟣", "Serious health risk on prolonged exposure."
    else:            return "Severe",        "#2c3e50", "⚫", "Affects healthy people severely."

def get_season(month):
    if   month in [12, 1, 2]:    return 0   # Winter
    elif month in [3, 4, 5]:     return 1   # Summer
    elif month in [6, 7, 8, 9]:  return 2   # Monsoon
    else:                         return 3   # Post-Monsoon

MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
DAY_NAMES   = ["Monday","Tuesday","Wednesday","Thursday",
               "Friday","Saturday","Sunday"]

In [8]:
with st.sidebar:
    st.markdown("## ⚙️ Input Parameters")
    st.markdown("---")

    st.markdown("### 📅 Date")
    month       = st.selectbox("Month", list(range(1,13)),
                               format_func=lambda x: MONTH_NAMES[x-1])
    day_of_week = st.selectbox("Day of Week", list(range(7)),
                               format_func=lambda x: DAY_NAMES[x])
    year        = st.selectbox("Year", [2022, 2023, 2024, 2025], index=2)
    city_enc    = st.slider("City Index", 0, 25, 5,
                            help="0–25 representing different cities")

    st.markdown("---")
    st.markdown("### 🧪 Pollutant Levels (μg/m³)")

    pm25  = st.slider("PM2.5",   0.0, 500.0, 85.0,  step=0.5)
    pm10  = st.slider("PM10",    0.0, 600.0, 160.0, step=0.5)
    no    = st.slider("NO",      0.0, 200.0, 18.0,  step=0.5)
    no2   = st.slider("NO2",     0.0, 200.0, 42.0,  step=0.5)
    nox   = st.slider("NOx",     0.0, 300.0, 55.0,  step=0.5)
    nh3   = st.slider("NH3",     0.0, 100.0, 14.0,  step=0.5)
    co    = st.slider("CO",      0.0,  10.0,  1.2,  step=0.1)
    so2   = st.slider("SO2",     0.0, 100.0, 12.0,  step=0.5)
    o3    = st.slider("O3",      0.0, 100.0, 28.0,  step=0.5)
    benz  = st.slider("Benzene", 0.0,  30.0,  2.5,  step=0.1)
    tol   = st.slider("Toluene", 0.0,  50.0,  4.0,  step=0.1)
    xyl   = st.slider("Xylene",  0.0,  20.0,  1.0,  step=0.1)

    st.markdown("---")
    predict_btn = st.button("🔮 Predict AQI", use_container_width=True)


# ─────────────────────────────────────────────────────────────
# Main Panel — Tabs
# ─────────────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["🔮 Prediction", "📊 Analysis", "📋 AQI Reference"])


# ══════════════════════════════════════════════
# TAB 1 — PREDICTION
# ══════════════════════════════════════════════
with tab1:
    if predict_btn:
        # Build input
        season_enc = get_season(month)
        input_df   = pd.DataFrame([{
            'PM2.5': pm25, 'PM10': pm10, 'NO': no,
            'NO2': no2, 'NOx': nox, 'NH3': nh3,
            'CO': co, 'SO2': so2, 'O3': o3,
            'Benzene': benz, 'Toluene': tol, 'Xylene': xyl,
            'month': month, 'day_of_week': day_of_week,
            'year': year, 'season_enc': season_enc,
            'City_enc': city_enc
        }])

        # Keep only model's expected features
        expected = list(model.feature_names_in_) if hasattr(model, 'feature_names_in_') \
                   else input_df.columns.tolist()
        input_df = input_df[[c for c in expected if c in input_df.columns]]

        pred_aqi = max(0, round(float(model.predict(input_df)[0]), 2))
        cat, color, emoji, health = get_aqi_info(pred_aqi)

        # ── AQI Result Card ───────────────────────────────────
        st.markdown(f"""
        <div class="aqi-card" style="background: linear-gradient(135deg, {color}cc, {color}66);
             border: 2px solid {color};">
            <div class="aqi-value">{pred_aqi:.0f}</div>
            <div class="aqi-label">{emoji} {cat}</div>
            <div style="font-size:0.95rem; margin-top:8px; opacity:0.9">{health}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("")

        # ── 4 Metric Boxes ────────────────────────────────────
        c1, c2, c3, c4 = st.columns(4)
        with c1:
            st.markdown(f"""
            <div class="metric-box">
                <div class="metric-val">{pred_aqi:.1f}</div>
                <div class="metric-lbl">AQI Value</div>
            </div>""", unsafe_allow_html=True)
        with c2:
            st.markdown(f"""
            <div class="metric-box">
                <div class="metric-val">{cat}</div>
                <div class="metric-lbl">Category</div>
            </div>""", unsafe_allow_html=True)
        with c3:
            st.markdown(f"""
            <div class="metric-box">
                <div class="metric-val">{MONTH_NAMES[month-1]}</div>
                <div class="metric-lbl">Month</div>
            </div>""", unsafe_allow_html=True)
        with c4:
            season_names = {0:"Winter ❄️", 1:"Summer ☀️",
                            2:"Monsoon 🌧️", 3:"Post-Monsoon 🍂"}
            st.markdown(f"""
            <div class="metric-box">
                <div class="metric-val" style="font-size:1.4rem">
                    {season_names[get_season(month)]}
                </div>
                <div class="metric-lbl">Season</div>
            </div>""", unsafe_allow_html=True)

        # ── AQI Gauge Bar ─────────────────────────────────────
        st.markdown('<p class="section-header">📏 AQI Scale</p>',
                    unsafe_allow_html=True)

        fig, ax = plt.subplots(figsize=(12, 1.2))
        fig.patch.set_facecolor('#0d0d1a')
        ax.set_facecolor('#0d0d1a')

        segments = [(50,"#2ecc71"),(100,"#f1c40f"),(200,"#e67e22"),
                    (300,"#e74c3c"),(400,"#8e44ad"),(500,"#2c3e50")]
        prev = 0
        for end, col in segments:
            ax.barh(0, end - prev, left=prev, height=0.6, color=col, alpha=0.85)
            prev = end

        # Needle
        ax.axvline(min(pred_aqi, 500), color='white', linewidth=3, zorder=5)
        ax.text(min(pred_aqi, 500), 0.45, f" {pred_aqi:.0f}",
                color='white', fontsize=11, fontweight='bold', va='center')

        ax.set_xlim(0, 500)
        ax.set_yticks([])
        ax.set_xticks([0, 50, 100, 200, 300, 400, 500])
        ax.tick_params(colors='white', labelsize=9)
        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout()
        st.pyplot(fig)
        plt.close()

    else:
        st.info("👈  Set the pollutant values in the sidebar and click **Predict AQI**")

        # Default AQI scale reference
        st.markdown('<p class="section-header">🌍 What is AQI?</p>',
                    unsafe_allow_html=True)
        st.markdown("""
        The **Air Quality Index (AQI)** is a number used to communicate how
        polluted the air is and what health effects might be a concern.
        A lower AQI value means cleaner air.

        | Range | Category | Health Impact |
        |-------|----------|---------------|
        | 0–50 | 🟢 Good | Minimal |
        | 51–100 | 🟡 Satisfactory | Minor for sensitive groups |
        | 101–200 | 🟠 Moderate | Discomfort for asthma patients |
        | 201–300 | 🔴 Poor | Breathing discomfort for most |
        | 301–400 | 🟣 Very Poor | Serious risk on prolonged exposure |
        | 401+ | ⚫ Severe | Affects healthy people too |
        """)


# ══════════════════════════════════════════════
# TAB 2 — ANALYSIS
# ══════════════════════════════════════════════
with tab2:
    st.markdown('<p class="section-header">🧪 Pollutant Input Overview</p>',
                unsafe_allow_html=True)

    pollutants = {
        "PM2.5": pm25, "PM10": pm10, "NO2": no2,
        "SO2":   so2,  "CO×10": co*10, "O3": o3,
        "NOx":   nox,  "NH3":  nh3
    }

    fig, ax = plt.subplots(figsize=(12, 4))
    fig.patch.set_facecolor('#0d0d1a')
    ax.set_facecolor('#0d0d1a')

    colors_bar = ["#e74c3c","#e67e22","#f1c40f","#2ecc71",
                  "#3498db","#9b59b6","#1abc9c","#e91e63"]
    bars = ax.bar(pollutants.keys(), pollutants.values(),
                  color=colors_bar, edgecolor='none', width=0.6)

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                f'{h:.1f}', ha='center', va='bottom',
                color='white', fontsize=9, fontweight='bold')

    ax.set_ylabel("Level (μg/m³)", color='white')
    ax.tick_params(colors='white')
    ax.set_title("Input Pollutant Levels", color='white', fontsize=13)
    for spine in ax.spines.values():
        spine.set_color('#2a2a4a')
    fig.tight_layout()
    st.pyplot(fig)
    plt.close()

    # ── Radar Chart ───────────────────────────────────────────
    st.markdown('<p class="section-header">🕸️ Pollutant Radar Chart</p>',
                unsafe_allow_html=True)

    radar_labels  = ["PM2.5","PM10","NO2","SO2","O3","NOx"]
    radar_max     = [500, 600, 200, 100, 100, 300]
    radar_vals    = [pm25, pm10, no2, so2, o3, nox]
    radar_norm    = [v/m for v, m in zip(radar_vals, radar_max)]
    radar_norm   += [radar_norm[0]]   # close polygon
    angles        = np.linspace(0, 2*np.pi, len(radar_labels),
                                endpoint=False).tolist()
    angles       += [angles[0]]

    fig, ax = plt.subplots(figsize=(5, 5),
                           subplot_kw=dict(polar=True))
    fig.patch.set_facecolor('#0d0d1a')
    ax.set_facecolor('#0d0d1a')

    ax.plot(angles, radar_norm, color='#38ef7d', linewidth=2)
    ax.fill(angles, radar_norm, color='#38ef7d', alpha=0.25)
    ax.set_thetagrids(np.degrees(angles[:-1]), radar_labels,
                      color='white', fontsize=10)
    ax.tick_params(colors='#555')
    ax.set_ylim(0, 1)
    ax.set_title("Normalised Pollutant Levels", color='white',
                 fontsize=12, pad=15)
    ax.spines['polar'].set_color('#2a2a4a')
    ax.yaxis.set_tick_params(labelcolor='#555')

    col_r, col_s = st.columns([1, 1])
    with col_r:
        st.pyplot(fig)
    with col_s:
        st.markdown("**Top Pollutants (your input):**")
        input_series = pd.Series(pollutants).sort_values(ascending=False)
        for name, val in input_series.items():
            st.markdown(f"- **{name}** : `{val:.1f} μg/m³`")
    plt.close()


# ══════════════════════════════════════════════
# TAB 3 — AQI REFERENCE
# ══════════════════════════════════════════════
with tab3:
    st.markdown('<p class="section-header">📋 AQI Category Reference</p>',
                unsafe_allow_html=True)

    ref_data = pd.DataFrame({
        "Emoji"     : ["🟢","🟡","🟠","🔴","🟣","⚫"],
        "Category"  : ["Good","Satisfactory","Moderate",
                       "Poor","Very Poor","Severe"],
        "AQI Range" : ["0 – 50","51 – 100","101 – 200",
                       "201 – 300","301 – 400","401+"],
        "Health Impact": [
            "Minimal impact",
            "Minor breathing discomfort to sensitive people",
            "Breathing discomfort to asthma patients & heart disease",
            "Breathing discomfort to most people on prolonged exposure",
            "Respiratory illness on prolonged exposure",
            "Affects healthy people; serious impact on sensitive people"
        ],
        "Recommended Action": [
            "No action needed — enjoy outdoor activities!",
            "Sensitive groups should reduce prolonged outdoor exertion",
            "People with lung/heart disease should limit outdoor activity",
            "Everyone should limit prolonged outdoor exertion",
            "Avoid outdoor activities; use air purifiers indoors",
            "Stay indoors; keep windows closed; use N95 masks outside"
        ]
    })
    st.dataframe(ref_data, use_container_width=True, hide_index=True)

    st.markdown('<p class="section-header">🔬 Key Pollutants Explained</p>',
                unsafe_allow_html=True)

    p1, p2 = st.columns(2)
    with p1:
        st.markdown("""
        **PM2.5** — Fine particulate matter < 2.5μm. Penetrates deep
        into lungs and bloodstream. Most harmful pollutant.

        **PM10** — Coarse particles < 10μm. Causes respiratory issues.

        **NO₂** — Nitrogen dioxide from vehicles & industry.
        Irritates airways.

        **SO₂** — Sulphur dioxide from burning fossil fuels.
        Causes acid rain.
        """)
    with p2:
        st.markdown("""
        **CO** — Carbon monoxide from incomplete combustion.
        Reduces oxygen in blood.

        **O₃** — Ground-level ozone. Formed by sunlight reacting
        with NOx. Lung irritant.

        **NH₃** — Ammonia from agriculture.
        Contributes to PM2.5 formation.

        **Benzene** — Carcinogenic VOC from fuel & industry.
        """)

# ─────────────────────────────────────────────────────────────
# Footer
# ─────────────────────────────────────────────────────────────
st.markdown("---")
st.caption("🌿 AQI Predictor · Built with Streamlit · Pragyan AI Hackathon · Problem #15")



2026-03-11 10:48:19.825 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.827 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-11 10:48:19.850 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [ ]:
!pip install streamlit pyngrok -q
print("✅ Streamlit & pyngrok installed!")


# ─────────────────────────────────────────────────────────────
# CELL 2: Upload app.py & model into Colab
# ─────────────────────────────────────────────────────────────
from google.colab import files

print("📂 Upload app.py and best_rf_aqi_model.pkl")
files.upload()   # select both files together


# ─────────────────────────────────────────────────────────────
# CELL 3: Verify files are present
# ─────────────────────────────────────────────────────────────
import os

for fname in ["app.py", "best_rf_aqi_model.pkl"]:
    if os.path.exists(fname):
        size = os.path.getsize(fname) / 1024
        print(f"  ✅ {fname:<30} ({size:.1f} KB)")
    else:
        print(f"  ❌ {fname:<30} NOT FOUND — upload again!")


# ─────────────────────────────────────────────────────────────
# CELL 4: 🚀 Launch Dashboard
# ─────────────────────────────────────────────────────────────
import subprocess
import time
from pyngrok import ngrok

# Kill any existing Streamlit processes
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Start Streamlit on port 8501
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false"],
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE
)

time.sleep(5)   # wait for server to start

# Create public URL with ngrok
public_url = ngrok.connect(8501)

print("\n" + "="*55)
print("🎉  YOUR DASHBOARD IS LIVE!")
print("="*55)
print(f"🌐  URL  :  {public_url}")
print("="*55)
print("\n👆 Click the link above to open your dashboard!")
print("   (Link stays active while this cell is running)")


# ─────────────────────────────────────────────────────────────
# CELL 5: Stop Dashboard (run when done)
# ─────────────────────────────────────────────────────────────
# ngrok.disconnect(public_url)
# proc.terminate()
# print("✅ Dashboard stopped.")



✅ Streamlit & pyngrok installed!
📂 Upload app.py and best_rf_aqi_model.pkl
